# 집계_1_총피해액
한국 수출기업 **전체**가 해협 위기로 더 낸 운임(KCCI 기준)과 수출 물량 변화를 월별로 계산합니다.

**추가 운임 부담 = 수출 적컨테이너(TEU) ÷ 2 × (그 달 평균 KCCI − 사건 전 12주 평균 KCCI) × 그 달 환율**

- 입력: `financedata/해수부_지역별_수출입컨테이너_월별.csv`, `financedata/KCCI.xls`, FRED 원/달러 환율(EXKOUS)
- 출력: `집계_1_총피해액.xlsx` (설명 / 위기별_요약 / 월별)

In [ ]:
from pathlib import Path
import pandas as pd

DESK = Path.home() / "OneDrive" / "Desktop"
DATA = DESK / "financedata"   # 입력 데이터
KEYS = DESK / "finance"       # API 키, KOSIS·관세청 원자료
SAVE = DESK / "finance"       # 결과 저장 위치 

import io
import requests

CRISES = {   # 위기: (해수부 지역, KCCI 항로, 사건일, 집계 시작월)
    "홍해": ("유럽 지역", ["KNEI", "KMDI"], "2023-11-19", "2023-12"),
    "호르무즈": ("중동 지역", ["KMEI"], "2026-02-28", "2026-03"),
}

## 1. 데이터 불러오기

In [ ]:
f = DATA / "해수부_지역별_수출입컨테이너_월별.csv"
try:
    c = pd.read_csv(f, dtype={"useYm": str, "연월": str})
except UnicodeDecodeError:                      
    c = pd.read_csv(f, dtype={"useYm": str, "연월": str}, encoding="cp949")
c = c.rename(columns={"useYm": "연월"})
c["월"] = pd.PeriodIndex(c["연월"], freq="M").astype(str)
c["수출적TEU"] = c["tFnshpFcontnTeu"] + c["tIntrvsslFcontnTeu"]

k = pd.read_excel(DATA / "KCCI.xls", header=1)
k["DATE"] = pd.to_datetime(k["DATE"].astype(str), format="%Y%m%d")
k = k.sort_values("DATE").set_index("DATE")
k["월"] = k.index.to_period("M").astype(str)

fx = pd.read_csv(io.StringIO(requests.get("https://fred.stlouisfed.org/graph/fredgraph.csv?id=EXKOUS", timeout=60).text))
fx.columns = ["date", "KRW_per_USD"]
fx["월"] = pd.to_datetime(fx["date"]).dt.to_period("M").astype(str)
fx = fx.set_index("월")["KRW_per_USD"]

## 2. 월별 추가 운임 부담

In [ ]:
monthly = []
for crisis, (area, routes, ev, start) in CRISES.items():
    ev = pd.Timestamp(ev)
    rate = k[routes].mean(axis=1)
    base = rate.loc[ev - pd.Timedelta(days=84): ev - pd.Timedelta(days=1)].mean()   # 평시 = 사건 전 12주
    rate_m = rate.groupby(k["월"]).mean()
    vol = c[c["areaNm"] == area].set_index("월")["수출적TEU"]
    for m in [m for m in vol.index if m >= start]:
        prev = str(pd.Period(m) - 12)
        fx_m = fx.get(m, fx.iloc[-1])
        extra = max(rate_m.get(m, float("nan")) - base, 0)
        feu = vol[m] / 2
        monthly.append({
            "위기": crisis, "지역": area, "월": m, "사건후_개월": (pd.Period(m) - pd.Period(start)).n + 1,
            "수출적컨테이너_TEU": round(vol[m]), "전년동월_TEU": round(vol.get(prev, float("nan"))),
            "물량_전년달 기준 증가 %": round((vol[m] / vol.get(prev) - 1) * 100, 1) if prev in vol else None,
            "운임비_월평균_USD": round(rate_m.get(m, float("nan"))),
            "사건 전 1분기 평균 운임비(평시 운임비)_USD": round(base),
            "추가운임비_FEU당_USD(컨테이너 1개당 더 낸 돈)": round(extra),
            "추가운임부담총액_백만USD": round(feu * extra / 1e6, 1),
            "환율_원per달러": round(fx_m, 1),
            "추가운임부담_억원": round(feu * extra * fx_m / 1e8, 0),
        })
m = pd.DataFrame(monthly)
m.head(6)

## 3. 위기별 요약 (첫 분기 = 사건 다음 달부터 3개월, 누적 = 자료 마지막 달까지)

In [ ]:
summary = []
for crisis, g in m.groupby("위기", sort=False):
    first = g[g["사건후_개월"] <= 3]
    summary.append({
        "위기": crisis, "지역": g["지역"].iloc[0],
        "평시운임비(사건 전 12주)_USD": g["사건 전 1분기 평균 운임비(평시 운임비)_USD"].iloc[0],
        "첫분기_기간": f"{first['월'].min()}~{first['월'].max()}",
        "첫분기_수출적컨테이너_TEU": int(first["수출적컨테이너_TEU"].sum()),
        "첫분기_평균추가운임비_FEU당_USD": int(first["추가운임비_FEU당_USD(컨테이너 1개당 더 낸 돈)"].mean()),
        "첫분기_추가운임부담_억원": int(first["추가운임부담_억원"].sum()),
        "첫분기_물량_전년대비비교%": round((first["수출적컨테이너_TEU"].sum() / first["전년동월_TEU"].sum() - 1) * 100, 1),
        "누적_기간": f"{g['월'].min()}~{g['월'].max()} ({len(g)}개월)",
        "누적_추가운임부담_억원": int(g["추가운임부담_억원"].sum()),
        "누적_추가운임부담_백만USD": round(g["추가운임부담총액_백만USD"].sum(), 0),
    })
s = pd.DataFrame(summary)
s.T

## 4. 저장

In [ ]:
notes = pd.DataFrame([
    ("목적", "해협 위기로 한국 수출기업 전체가 부담한 추가 운임과 수출 물량 감소를 집계"),
    ("수출 컨테이너", "해양수산부 수출입컨테이너처리실적(공공데이터포털 API), 지역별·월별 수출 적컨테이너 TEU(국적선+외국선)"),
    ("운임비", "KCCI 주간 → 월평균. 유럽 = KNEI·KMDI 평균, 중동 = KMEI. 평시 = 사건 전 12주 평균"),
    ("추가 운임 부담", "수출 적컨테이너 TEU ÷ 2(FEU 환산) × max(그 달 운임비 − 평시, 0)"),
    ("환율", "FRED EXKOUS 월평균(원/달러). 미발표 달은 최근 값 사용"),
    ("물량 전년比", "수출 적컨테이너의 전년 같은 달 대비 증감률"),
    ("한계", "KCCI는 부산발 현물(spot) 운임 → 장기계약 화주는 인상이 늦게·작게 반영되어 실제보다 크게 잡힐 수 있음. "
            "반대로 할증료·지연 비용·물량 감소로 잃은 매출은 포함 안 됨. 운임 변화 전부를 위기 탓으로 봄(시장 요인 미분리)"),
], columns=["항목", "내용"])
out = SAVE / "집계_1_총피해액.xlsx"
with pd.ExcelWriter(out) as w:
    notes.to_excel(w, sheet_name="설명", index=False)
    s.to_excel(w, sheet_name="위기별_요약", index=False)
    m.to_excel(w, sheet_name="월별", index=False)
print("저장:", out)